# What is YOLO, and why use it?

YOLO in AI stands for **You Only Look Once.** It's a popular algorithm used in **computer vision** for real-time **object detection**.

Simple idea...

Traditional methods look at an image many times to find objects.

**YOLO looks at the image only once** and detects everything in one go -> faster and efficient.

...

**Audience.** AI students who have used Keras / PyTorch / TensorFlow and want to understand what YOLO actually is, what problem it solves, and when to reach for it. Engineers will also find the *when-not-to* and *deployment* sections useful.

**This notebook is mostly conceptual** - no training, no big code. It exists to give you the mental model before you touch the API.

## Table of contents
1. The one-sentence pitch
2. The original idea: "You Only Look Once"
3. One-stage vs two-stage detectors - the speed/accuracy axis
4. The YOLO family timeline (2015 -> 2026)
5. The five tasks YOLO can do (yes, including classification)
6. Why people love it - and when it's the wrong tool
7. The Ultralytics package - the modern way to use YOLO
8. Model size variants: how to pick `n / s / m / l / x`
9. What's new in YOLO26 (Jan 2026)
10. Where to go next


## 1. The one-sentence pitch

> **YOLO** is a single neural network that, given an image, predicts *what is in it and where* in **one forward pass** - fast enough to run on a phone, accurate enough for production.

That's the whole hook. Before YOLO (2015), the dominant detection approach (R-CNN, Fast R-CNN, Faster R-CNN) was *two-stage*: first propose regions of interest, then classify each region. That's slow. YOLO collapses both stages into one network. Hence **You Only Look Once**.


or

The key idea is in the name: instead of scanning an image multiple times with sliding windows or running a separate region-proposal step (like older methods such as R-CNN did), YOLO passes the image through a single neural network just *once*. That network simultaneously predicts:

- **Bounding boxes** (where objects are)
- **Class labels** (what each object is — person, car, drone, etc.)
- **Confidence scores** (how sure the model is)

It does this by dividing the image into a grid, where each grid cell is responsible for predicting objects whose center falls inside it.

**Why it matters:** Because everything happens in one forward pass, YOLO is extremely fast — fast enough for real-time applications like self-driving cars, video surveillance, drone navigation, sports analytics, and robotics. Earlier detectors were accurate but too slow for live video.

## 2. The original idea (YOLOv1, 2015)

In 2015 was to **frame detection as a single regression problem**:

- Divide the image into an `S × S` grid (e.g. 7×7).
- For each grid cell, the network outputs:
  - `B` bounding boxes with `(x, y, w, h, confidence)`.
  - One vector of class probabilities.
- Train it end-to-end with one combined loss.

Compared to the two-stage pipelines of the day, YOLOv1 was an order of magnitude faster (45 FPS on a Titan X) at modest accuracy cost. Every YOLO since has refined that one idea — better backbones, anchor boxes, multi-scale heads, anchor-free heads, and most recently NMS-free heads.


## 3. One-stage vs two-stage detectors

| property               | two-stage (Faster R-CNN family)            | one-stage (YOLO, SSD, RetinaNet)            |
|------------------------|--------------------------------------------|---------------------------------------------|
| pipeline               | propose regions → classify each            | predict everything in one forward pass      |
| typical accuracy       | very high                                  | high (gap has nearly closed since 2020)     |
| typical speed          | tens of ms/image                           | a few ms/image                              |
| deployment complexity  | higher — multiple subnets, NMS             | lower — single net, NMS-free in YOLO26      |
| best fit               | offline analytics, max-accuracy benchmarks | real-time, edge, embedded, robotics, video  |

Two-stage models are still standard in some niches (medical imaging, certain remote-sensing tasks). For *real-time anything*, one-stage models — and especially YOLO — dominate.


## 4. The YOLO timeline

Simplified, just the milestones:

| year | version       | author / org                | landmark idea                                                         |
|------|---------------|-----------------------------|------------------------------------------------------------------------|
| 2015 | YOLOv1        | Joseph Redmon               | First single-stage detector worth its name.                            |
| 2016 | YOLOv2/9000   | Redmon                      | Anchor boxes, batch norm, multi-scale training.                        |
| 2018 | YOLOv3        | Redmon                      | Multi-scale heads (3 scales), Darknet-53 backbone.                     |
| 2020 | YOLOv4        | Bochkovskiy et al.          | Mosaic augmentation, CSPDarknet, lots of "bag of tricks".              |
| 2020 | YOLOv5        | Ultralytics (Glenn Jocher)  | Pure-PyTorch reimplementation. Made YOLO mainstream in industry.       |
| 2022 | YOLOv6/v7     | Meituan / WongKinYiu        | Architecture refinements, deployable variants.                         |
| 2023 | YOLOv8        | Ultralytics                 | Anchor-free heads, decoupled detection head, all five tasks unified.   |
| 2024 | YOLOv10       | Tsinghua (Wang et al.)      | First **NMS-free** end-to-end YOLO.                                    |
| 2024 | YOLO11        | Ultralytics                 | Hybrid task assignment, efficiency modules.                            |
| 2026 | **YOLO26**    | Ultralytics                 | DFL removal, native NMS-free, ProgLoss+STAL, MuSGD optimizer.          |

**Two practical takeaways:**
1. The name *YOLO* is shared across many independent projects; only the **Ultralytics** ones (`v5`, `v8`, `11`, `26`) are the same codebase.
2. Two versions stand out for production today: **YOLO11** (mature, very stable) and **YOLO26** (latest, faster on edge).


## 5. Five tasks YOLO can do

Modern Ultralytics YOLO is not just a detector — the same backbone can produce:

| task                       | input  | output                                       | typical use                       |
|----------------------------|--------|----------------------------------------------|-----------------------------------|
| Object detection           | image  | list of `(box, class, conf)`                 | counting, surveillance, robotics  |
| Instance segmentation      | image  | list of `(mask, class, conf)`                | pixel-precise vision, AR, medical |
| **Image classification**   | image  | one class per image                          | sorting, quality control          |
| Pose estimation            | image  | list of person + keypoints                   | sports analytics, AR, fitness     |
| Oriented bounding boxes    | image  | list of *rotated* boxes                      | aerial / satellite imagery        |

**Naming convention.** A task is encoded in the model filename:

| filename            | task              |
|---------------------|-------------------|
| `yolo26n.pt`        | detection         |
| `yolo26n-seg.pt`    | segmentation      |
| `yolo26n-cls.pt`    | classification    |
| `yolo26n-pose.pt`   | pose              |
| `yolo26n-obb.pt`    | oriented boxes    |

Notice that the original notebook in this project loads **`yolo26n-cls.pt`** — the classification head. Notebook 1 in this series goes deep on that.


## 6. Why people love YOLO — and when *not* to use it

**Strengths:**
1. **Speed.** A YOLO26-nano runs at hundreds of FPS on a modern GPU and 30+ FPS on a phone CPU.
2. **One API for five tasks.** `YOLO('yolo26n.pt')` and `model.train(...)` work the same whether you're doing detection, segmentation, or classification.
3. **Production-ready exports.** ONNX, TensorRT, CoreML, TFLite, OpenVINO out of the box.
4. **Strong defaults.** Augmentation, learning-rate schedule, anchor-free heads, NMS-free output — you rarely need to touch these.
5. **Massive community.** Thousands of tutorials, datasets formatted for YOLO already exist.

**When YOLO is the wrong tool:**
1. **Per-pixel semantic segmentation** of complex scenes (cityscape labels, medical organ maps). Use DeepLab, Mask2Former, or SAM-style models.
2. **Very tiny objects** in very large images (small <8×8 px in a 4K image). Specialised architectures or tiling pipelines do better — YOLO26 has improved this with `STAL` but it's still not magic.
3. **Pure image-level classification with millions of training images.** A dedicated CNN (EfficientNet, ConvNeXt) or ViT is usually a hair more accurate per parameter, though YOLO-cls is competitive.
4. **You need explainability / saliency maps as part of the contract.** YOLO outputs are easy to use but harder to interpret than two-stage approaches.


## 7. The Ultralytics package — the modern way

Since YOLOv5, the canonical implementation has been the `ultralytics` Python package. Install it with one line:

```bash
uv add ultralytics
```

The whole API is essentially three calls:

```python
from ultralytics import YOLO

model = YOLO('yolo26n.pt')                              # 1. load
model.train(data='my-dataset.yaml', epochs=100)         # 2. train (or fine-tune)
results = model('image.jpg')                            # 3. predict
```

There's also a CLI (`yolo train ...`, `yolo predict ...`) for shell-script users.

**Why this matters for AI engineers.** You spend more time on data and deployment than on model code. The Ultralytics API standardises the model code so the rest of your time is freed up.


## 8. Picking a model size: n / s / m / l / x

Every Ultralytics YOLO ships in **five sizes**. They're the same architecture scaled up.

**YOLO26 classification (ImageNet top-1, fused params):**

| variant         | top-1 acc | size (MB) | when to pick it                                         |
|-----------------|-----------|-----------|----------------------------------------------------------|
| `yolo26n-cls`   | 71.4%     | ~6        | phones, microcontrollers, real-time on CPU              |
| `yolo26s-cls`   | 76.0%     | ~13       | web app on CPU, jetson nano                              |
| `yolo26m-cls`   | 78.1%     | ~22       | balanced default for most projects                       |
| `yolo26l-cls`   | 79.0%     | ~27       | server-side, accuracy matters more than speed            |
| `yolo26x-cls`   | 79.9%     | ~57       | benchmark / leaderboard, accuracy maxed                  |

**Heuristic.** Start with `n` to debug your pipeline, then switch to `m` for the real run, then upgrade only if metrics demand it.

**Trade-off rule of thumb.** Going from `n` → `x` typically buys you ~6-10 absolute accuracy points at ~10× the latency. Diminishing returns kick in after `m` for most real-world datasets.


## 9. What's new in YOLO26 (released Jan 14, 2026)

YOLO26 is the latest Ultralytics release. The headline changes are aimed at *edge deployment*: smaller, simpler, faster on CPU.

| change                              | what it means                                                                                              |
|-------------------------------------|------------------------------------------------------------------------------------------------------------|
| **DFL removal**                     | Distribution Focal Loss is gone from the head. Simpler graph → cleaner ONNX/TFLite exports.                |
| **End-to-end NMS-free inference**   | Detection head outputs final boxes directly — no post-processing step. (Pioneered by YOLOv10, refined here.)|
| **ProgLoss**                        | *Progressive* loss balancing: re-weights detection terms as training progresses. Better convergence.       |
| **STAL**                            | *Small-Target-Aware Label assignment* — fixes the famous YOLO weak spot on tiny objects.                   |
| **MuSGD optimizer**                 | Hybrid of SGD + Muon (the optimizer behind some recent LLMs). More stable training, better final loss.     |
| **Up to 43% faster CPU inference**  | Concrete win for IoT, robotics, in-browser deployment.                                                     |

**Important caveat for this project.** We're doing **classification**, not detection. NMS-free and ProgLoss+STAL are *detection* features — they don't apply to `yolo26-cls`. What you *do* benefit from in classification: the simpler architecture, faster CPU inference, and the MuSGD optimizer.


## 10. What to read / try next

Now that you have the mental model, here is a recommended path through the rest of this notebook series:

1. **`yolo_img_classifications.ipynb`** — Image classification with YOLO26 explained step by step, including the dataset format, training arguments, validation metrics, and exports.
2. **`yolo-v1.ipynb`** — A clean, runnable training notebook for our Rock-Paper-Scissors dataset.
3. **`test-yolo.ipynb`** — A clean inference notebook with visualisations and batch prediction.

Useful external references:
- Ultralytics docs: <https://docs.ultralytics.com/>
- YOLO26 model card: <https://docs.ultralytics.com/models/yolo26/>
- Classification task: <https://docs.ultralytics.com/tasks/classify/>
- ArXiv overview paper (2026): *Ultralytics YOLO Evolution* (Sapkota & Karkee, arXiv:2510.09653).
